<a href="https://colab.research.google.com/github/Alshihab941/analysis-of-dna-sequences-with-python/blob/main/research_dna_analyzer.py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install biopython

In [7]:
from collections import Counter
from math import log2
from Bio import SeqIO

# Create a dummy FASTA file for demonstration purposes
# In a real scenario, you would upload your own sequences.fasta file.
with open('sequences.fasta', 'w') as f:
    f.write('>seq1\nATGCGTACGTACGTAGCTAGCTAGCTAGCTACGTAGCTACGTAGCATGCATGCA\n')
    f.write('>seq2\nGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCA\n')
    f.write('>seq3\nATATATATATGCGCGCGCGCTATATATATATGCGCGCGCGC\n')


def validat_dna(sequence):
  """Return True if the sequence contains only A, T, C, and G."""
  sequence = sequence.upper()
  return set(sequence).issubset({"A", "T", "C", "G"})


def nucleotide_composition(sequence):
  """Calculate nucleotide counts and percentages."""
  sequence = sequence.upper() # Fixed typo: squence to sequence
  length = len(sequence)

  counts = Counter(sequence)

  results = {}


  for base in "ATCG":
    count = counts.get(base,0)
    percentage = (count / length) * 100 if length > 0 else 0
    results[base] = {
        "count": count,
        "percentage": percentage
    }

  return results


def gc_content(sequence):
  """Calculate GC content."""
  sequence = sequence.upper()

  if len(sequence) == 0:
    return 0

  gc = sequence.count("G") + sequence.count("C")

  return(gc / len(sequence)) * 100

def calculate_kmers(sequence, k):
  """Count k-mers of length K."""
  sequence = sequence.upper()

  return  Counter(
      sequence[i:i+k] # Fixed typo: seqeunce to sequence
      for i in range (len(sequence) - k + 1)

      )

def nucleotide_entropy(sequence):
    """Calculate  Shanon entropy of nucleotide frequencies."""
    sequence = sequence.upper()

    counts = Counter(sequence)
    total = len(sequence)
    entropy = 0 # Initialized entropy

    if total == 0:
      return 0

    for count in counts.values():
      probability = count / total
      entropy -= probability * log2(probability)

    return entropy # Moved return statement to the end


def analyze_sequence(record): # Renamed 'codon_composition' to 'analyze_sequence' and changed argument to 'record'
    """Analyze one Biopython SeqRecord."""

    dna_sequence = str(record.seq).upper() # Using dna_sequence to avoid confusion with 'record'

    if not validat_dna(dna_sequence): # Fixed typo: validat_dna to validate_dna
      raise ValueError(
          f"{record.id} contains invalid DNA characters."

      )

    composition = nucleotide_composition(dna_sequence)
    gc = gc_content(dna_sequence)
    kmers = calculate_kmers(dna_sequence, k=3)
    entropy = nucleotide_entropy(dna_sequence)

    return{
       "id": record.id,
       "length": len(dna_sequence), # Fixed typo: lenght to length
       "composition": composition,
       "gc_content": gc,
       "top_kmers": kmers.most_common(5), # Fixed typo: most_commom to most_common
       "entropy": entropy

       }

def analyze_fasta(filename):
  """Read and analyze all sequences in a Fasta file."""
  records = list(SeqIO.parse(filename, "fasta"))

  print("-" * 70)
  print("DNA SEQUENCE ANALYSIS")
  print("-" * 70)

  print(f"Number of sequences: {len(records)}")

  for record in records:

    result = analyze_sequence(record) # Call analyze_sequence

    print("\nSequence:", result["id"])
    print("Length:", result["length"], "bp")

    print("Nucleotide composition:")

    for base in "ATCG":
      data = result["composition"][base]

      print(
          f" {base}: "
          f"{data['count']}"
          f"({data['percentage']:.2f}%)"

      )

    # Corrected indentation for these print statements
    print(
        f"GC content: "
        f"{result['gc_content']:.2f}%"
    )

    print(
        "Top 3-mers:",
        result["top_kmers"] # Fixed typo: top_Kmers to top_kmers
    )

    print(
        f"Nucleotide Shannon entropy: " # Fixed typo: Shanon to Shannon
        f"{result['entropy']:.4f}"
    )

if __name__ == "__main__": # Fixed typo: _name_ to __name__
  analyze_fasta("sequences.fasta")


----------------------------------------------------------------------
DNA SEQUENCE ANALYSIS
----------------------------------------------------------------------
Number of sequences: 3

Sequence: seq1
Length: 54 bp
Nucleotide composition:
 A: 14(25.93%)
 T: 13(24.07%)
 C: 13(24.07%)
 G: 14(25.93%)
GC content: 50.00%
Top 3-mers: [('TAG', 6), ('AGC', 6), ('CGT', 5), ('GTA', 5), ('GCT', 5)]
Nucleotide Shannon entropy: 1.9990

Sequence: seq2
Length: 47 bp
Nucleotide composition:
 A: 12(25.53%)
 T: 11(23.40%)
 C: 12(25.53%)
 G: 12(25.53%)
GC content: 51.06%
Top 3-mers: [('GCA', 12), ('CAT', 11), ('ATG', 11), ('TGC', 11)]
Nucleotide Shannon entropy: 1.9990

Sequence: seq3
Length: 41 bp
Nucleotide composition:
 A: 10(24.39%)
 T: 11(26.83%)
 C: 10(24.39%)
 G: 10(24.39%)
GC content: 48.78%
Top 3-mers: [('TAT', 9), ('ATA', 8), ('GCG', 8), ('CGC', 8), ('ATG', 2)]
Nucleotide Shannon entropy: 1.9987
